### Section 0
## Loaing Necessary Python Libraries

In [2]:
pip install astroquery -q

Note: you may need to restart the kernel to use updated packages.


In [3]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.table import Table
from astropy.coordinates import SkyCoord
import astropy.units as u
import astropy.units as u
from astropy.coordinates import SkyCoord
from astroquery.vizier import Vizier
from astroquery.xmatch import XMatch
from astroquery.simbad import Simbad
import requests
from alerce.core import Alerce
client = Alerce()

FITS_PATH    = "BAT_catalog.fits"

### Section 1
## Understanding the FITS structure

In [4]:
# STEP 1: Read the FITS structure
fits_path = FITS_PATH

with fits.open(fits_path) as hdul:
    print("--- FITS Extensions Structure ---")
    hdul.info()

--- FITS Extensions Structure ---
Filename: BAT_catalog.fits
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU       6   ()      
  1  INPUT_CATALOG    1 BinTableHDU   1440   923R x 6C   [30A, 30A, 1J, 1E, 1E, 30A]   


In [5]:
bat_table = Table.read(fits_path, hdu=1)
bat_df = bat_table.to_pandas()
df=bat_df 
# Preview the column names and the first 5 rows
print("Columns in BAT catalog:", bat_df.columns.tolist())
bat_df.head()

Columns in BAT catalog: ['NAME', 'ALT_NAME', 'CATNUM', 'RA_OBJ', 'DEC_OBJ', 'SRC_TYPE']


,NAME,ALT_NAME,CATNUM,RA_OBJ,DEC_OBJ,SRC_TYPE
0,b'QSO B0003-066 ',b'NRAO 005 ',33001,1.557887,-6.393149,b'BL Lac '
1,b'PSR J0007+7303 ',b'0FGL J0007.4+7303 ',33730,1.756000,73.052002,b'Pulsar '
2,b'QSO B0007+107 ',b'III Zw 2 ',33002,2.629191,10.974862,b'Sy1 '
3,b'QSO B0016+73 ',NaN,33003,4.940777,73.458336,b'Quasar '
4,b'IGR J00234+6141 ',b'2MASS J00225764+6141075 ',10598,5.850000,61.692001,b'CV '


In [7]:
for col in bat_df.columns:
    # If the column contains byte strings, decode them to standard text strings
    if bat_df[col].dtype == object:
        bat_df[col] = bat_df[col].apply(lambda x: x.decode('utf-8').strip() if isinstance(x, bytes) else (x.strip() if isinstance(x, str) else x))

# Show the cleaned result
print("Cleaned column columns check:")
print(bat_df[['NAME','ALT_NAME', 'RA_OBJ', 'DEC_OBJ']].head())

Cleaned column columns check:
              NAME                 ALT_NAME    RA_OBJ    DEC_OBJ
0    QSO B0003-066                 NRAO 005  1.557887  -6.393149
1   PSR J0007+7303        0FGL J0007.4+7303  1.756000  73.052002
2    QSO B0007+107                 III Zw 2  2.629191  10.974862
3     QSO B0016+73                      NaN  4.940777  73.458336
4  IGR J00234+6141  2MASS J00225764+6141075  5.850000  61.692001


### Section 2
## Searching Maxi

In [8]:
def ra_dec_to_maxi_id(ra_deg: float, dec_deg: float) -> str:
    """
    Convert decimal RA/Dec degrees to a MAXI J-name string.Format: J + HHMM + sign + DDd
      HHMM  = zero-padded hours (HH) and minutes (MM), truncated (not rounded)
      DDd   = absolute degrees (DD) and tenths (d), truncated
      sign  = '+' or '-'
    """
    # RA: degrees → total hours, then split into HH and MM 
    ra_hours_total = ra_deg / 15.0          # 1 hour = 15 degrees
    hh = int(ra_hours_total)               # integer hours
    mm = int((ra_hours_total - hh) * 60)   # integer minutes (truncated)
    ra_str = f'{hh:02d}{mm:02d}'           

    # Dec: degrees → sign, integer degrees, and tenths 
    sign = '+' if dec_deg >= 0 else '-'
    abs_dec = abs(dec_deg)
    dd = int(abs_dec)                      # integer degrees
    d  = int((abs_dec - dd) * 10)          # tenths of a degree (truncated)
    dec_str = f'{dd:02d}{d}'               

    return f'J{ra_str}{sign}{dec_str}'     
    

###### Self Note: Python Type Hints & Function Architecture

The syntax `(ra_deg: float, dec_deg: float) -> str` uses **Python Type Hints** 

* **What it do:** it explicitly declare the expected data types for the inputs (`float` variables for coordinates) and the final output (`-> str` means the function will return a text string).
* **Why we use it:** Type hints do not change how the code runs dynamically, but they provide self-documentation, prevent bugs.

In [9]:
# --- Main MAXI check loop ---
MAXI_BASE = 'https://maxi.riken.jp/star_data'
MAXI_DELAY = 0.3   # seconds between requests (be polite to the server)
MAXI_TIMEOUT = 5   # seconds before giving up on a single request

# 1. ADD THIS: A custom User-Agent to disguise the request as a normal browser
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36'
}

maxi_matches = []  # will hold True/False for each source

for i, row in df.iterrows():
    source_name = row['NAME']
    ra          = row['RA_OBJ']
    dec         = row['DEC_OBJ']

    # Build the MAXI J-name and URL
    maxi_id  = ra_dec_to_maxi_id(ra, dec)
    maxi_url = f'{MAXI_BASE}/{maxi_id}/{maxi_id}.html'

    try:
        # 2. UPDATE THIS: Pass the HEADERS dictionary into your request
        resp = requests.get(maxi_url, headers=HEADERS, timeout=MAXI_TIMEOUT)
        matched = (resp.status_code == 200)
    except Exception as exc:
        # Network error, DNS failure, timeout, etc.
        print(f'  [WARNING] MAXI request failed for {source_name}: {exc}')
        matched = False

    maxi_matches.append(matched)

    # Progress report every 50 sources (1-indexed)
    if (i + 1) % 50 == 0:
        print(f'Checking MAXI: {i + 1}/{len(df)}...')

    time.sleep(MAXI_DELAY)  # polite delay

# Store results
df['MAXI_match'] = maxi_matches

print(f'\nMAXI check complete. Matches: {sum(maxi_matches)}/{len(df)}')

Checking MAXI: 50/923...
Checking MAXI: 100/923...
Checking MAXI: 150/923...
Checking MAXI: 200/923...
Checking MAXI: 250/923...
Checking MAXI: 300/923...
Checking MAXI: 350/923...
Checking MAXI: 400/923...
Checking MAXI: 450/923...
  [WARNING] MAXI request failed for TrA X-1: HTTPSConnectionPool(host='maxi.riken.jp', port=443): Max retries exceeded with url: /star_data/J1528-618/J1528-618.html (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x10dbd6870>, 'Connection to maxi.riken.jp timed out. (connect timeout=5)'))
  [WARNING] MAXI request failed for QSO B1533+535: HTTPSConnectionPool(host='maxi.riken.jp', port=443): Max retries exceeded with url: /star_data/J1535+533/J1535+533.html (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x10dbda0c0>, 'Connection to maxi.riken.jp timed out. (connect timeout=5)'))
  [WARNING] MAXI request failed for IGR J15360-5750: HTTPSConnectionPool(host='maxi.riken.jp', port=443): Read timed out. (

### Section 3
## Saving the matches in a CSV file

In [10]:
# Filter to get only the 270 sources that have a MAXI match
matched_df = bat_df[bat_df['MAXI_match'] == True].copy()


# Defining  unpacking function i
def get_maxi_id_from_row(row):
    """Takes a single pandas row, extracts RA/Dec, and returns the MAXI ID."""
    ra = row['RA_OBJ']
    dec = row['DEC_OBJ']
    
    # Calculate the ID using our main function
    maxi_id = ra_dec_to_maxi_id(ra, dec)
    
    return maxi_id


#  Applying explicit function to every row across the dataframe (axis=1)
matched_df['MAXI_NAME'] = matched_df.apply(get_maxi_id_from_row, axis=1)


# Selecting and renaming only the columns we want in our table
csv_export_df = matched_df[[
    'NAME',     # BAT catalog name
    'RA_OBJ',   # RA from BAT
    'DEC_OBJ',  #  Dec from BAT
    'MAXI_NAME' # The matching MAXI web ID
]].copy()

csv_export_df.columns = ['BAT_Catalogue_Name', 'RA_BAT', 'DEC_BAT', 'MAXI_Name']


# Saving the local CSV file
output_filename = "BAT_MAXI_matched_sources_try2.csv"
csv_export_df.to_csv(output_filename, index=False)

print(f" Successfully saved {len(csv_export_df)} matched sources to: {output_filename}")
csv_export_df.head()

 Successfully saved 271 matched sources to: BAT_MAXI_matched_sources_try2.csv


,BAT_Catalogue_Name,RA_BAT,DEC_BAT,MAXI_Name
6,V709 Cas,7.20362,59.289398,J0028+592
7,IGR J00291+5934,7.26300,59.571999,J0029+595
12,QSO B0033+595,8.96929,59.834599,J0035+598
13,IGR J00370+6122,9.25000,61.367001,J0037+613
17,NGC 262,12.19640,31.957001,J0048+319


#### Now I am going to use the SIMBAD first method:

In [17]:
import pandas as pd
from astroquery.simbad import Simbad
import time

# 1. Load the MAXI-matched CSV
csv_path = "BAT_MAXI_matched_sources_try2.csv"
matched_df = pd.read_csv(csv_path)
print(f"Loaded {len(matched_df)} sources from: {csv_path}")

# 2. Configure SIMBAD
custom_simbad = Simbad()
custom_simbad.add_votable_fields('otype')  # ra/dec already returned by default in degrees

# 3. Result lists
simbad_main_ids = []
simbad_otypes   = []
simbad_ras      = []
simbad_decs     = []

print("\nQuerying SIMBAD row-by-row for precise optical coordinates...")

# 4. Loop
for index, row in matched_df.iterrows():
    name_to_search = row['BAT_Catalogue_Name']
    try:
        result_table = custom_simbad.query_object(name_to_search)
        if result_table is not None and len(result_table) > 0:
            cols = result_table.colnames

            # Main ID
            if 'main_id' in cols:   raw_main_id = result_table['main_id'][0]
            elif 'MAIN_ID' in cols: raw_main_id = result_table['MAIN_ID'][0]
            else:                   raw_main_id = None

            # Otype
            if 'otype' in cols:   raw_otype = result_table['otype'][0]
            elif 'OTYPE' in cols: raw_otype = result_table['OTYPE'][0]
            else:                 raw_otype = None

            # RA (degrees)
            if 'ra' in cols:   raw_ra = float(result_table['ra'][0])
            elif 'RA' in cols: raw_ra = float(result_table['RA'][0])
            else:              raw_ra = None

            # Dec (degrees)
            if 'dec' in cols:   raw_dec = float(result_table['dec'][0])
            elif 'DEC' in cols: raw_dec = float(result_table['DEC'][0])
            else:               raw_dec = None

            # Clean text fields
            clean_main_id = raw_main_id.decode('utf-8').strip() if isinstance(raw_main_id, bytes) else (str(raw_main_id).strip() if raw_main_id is not None else None)
            clean_otype   = raw_otype.decode('utf-8').strip()   if isinstance(raw_otype,   bytes) else (str(raw_otype).strip()   if raw_otype   is not None else None)

            simbad_main_ids.append(clean_main_id)
            simbad_otypes.append(clean_otype)
            simbad_ras.append(raw_ra)
            simbad_decs.append(raw_dec)

        else:
            simbad_main_ids.append(None)
            simbad_otypes.append(None)
            simbad_ras.append(None)
            simbad_decs.append(None)

    except Exception as error:
        print(f"  [SKIP] {name_to_search}: {error}")
        simbad_main_ids.append(None)
        simbad_otypes.append(None)
        simbad_ras.append(None)
        simbad_decs.append(None)

    if (index + 1) % 30 == 0:
        print(f"  Progress: {index + 1}/{len(matched_df)}...")
    time.sleep(0.1)

# 5. Attach columns
matched_df['SIMBAD_ID']    = simbad_main_ids
matched_df['SIMBAD_Otype'] = simbad_otypes
matched_df['SIMBAD_RA']    = simbad_ras
matched_df['SIMBAD_DEC']   = simbad_decs

# 6. Save
output_path = "BAT_MAXI_SIMBAD_precise_coords_copy.csv"
matched_df.to_csv(output_path, index=False)
print(f"\nDone. Saved {len(matched_df)} sources with precise coords → {output_path}")

matched_df[['BAT_Catalogue_Name', 'SIMBAD_ID', 'SIMBAD_Otype', 'SIMBAD_RA', 'SIMBAD_DEC']].head(10)

Loaded 271 sources from: BAT_MAXI_matched_sources_try2.csv

Querying SIMBAD row-by-row for precise optical coordinates...
  Progress: 30/271...
  Progress: 60/271...
  Progress: 90/271...


  Progress: 120/271...
  Progress: 150/271...
  Progress: 180/271...
  Progress: 210/271...
  Progress: 240/271...
  Progress: 270/271...

Done. Saved 271 sources with precise coords → BAT_MAXI_SIMBAD_precise_coords_copy.csv


,BAT_Catalogue_Name,SIMBAD_ID,SIMBAD_Otype,SIMBAD_RA,SIMBAD_DEC
0,V709 Cas,V* V709 Cas,CV*,7.203445,59.289454
1,IGR J00291+5934,V* V1037 Cas,LXB,7.262750,59.571944
2,QSO B0033+595,RX J0035.8+5950,BLL,8.969353,59.834587
3,IGR J00370+6122,BD+60 73,HXB,9.290149,61.360136
4,NGC 262,NGC 262,Bla,12.196423,31.956968
5,Mrk 1148,Mrk 1148,Sy1,12.978181,17.432919
6,SMC X-3,LIN 198,HXB,13.023438,-72.434508
7,Gam Cas,* gam Cas,HXB,14.177213,60.716740
8,SMC X-1,SK 160,HXB,19.271441,-73.443337
9,3A 0114+650,V* V662 Cas,HXB,19.511240,65.291619


In [16]:
result_table = custom_simbad.query_object("Aql X-1")
print(result_table.colnames)
print(result_table)

['main_id', 'ra', 'dec', 'coo_err_maj', 'coo_err_min', 'coo_err_angle', 'coo_wavelength', 'coo_bibcode', 'otype', 'matched_id']
 main_id         ra              dec         ... otype matched_id
                deg              deg         ...                 
--------- --------------- ------------------ ... ----- ----------
X Aql X-1 287.81690471389 0.5849633979699999 ...   LXB  X Aql X-1


In [19]:
client = Alerce()

# 1. Load the CSV containing our hyper-precise SIMBAD coordinates
input_path = "BAT_MAXI_SIMBAD_precise_coords_copy.csv"
df = pd.read_csv(input_path)
print(f"Loaded {len(df)} sources from: {input_path}")

# Filter out rows where SIMBAD couldn't resolve coordinates
valid_df = df.dropna(subset=['SIMBAD_RA', 'SIMBAD_DEC']).copy()
print(f"Found {len(valid_df)} sources with valid coordinates to target on ZTF.")

# 2. Storage arrays
ztf_ids    = []
ztf_ras    = []
ztf_decs   = []
separations = []

CONE_RADIUS_ARCSEC = 5.0
print(f"\nStarting precise ZTF cross-match (Radius: {CONE_RADIUS_ARCSEC} arcseconds)...")

# 3. Core matching loop
for index, row in valid_df.iterrows():
    name      = row['BAT_Catalogue_Name']
    ra_simbad  = row['SIMBAD_RA']
    dec_simbad = row['SIMBAD_DEC']

    center_coord = SkyCoord(ra=ra_simbad, dec=dec_simbad, unit=(u.deg, u.deg), frame='icrs')

    try:
        matches = client.query_objects(
            survey="ztf",
            ra=ra_simbad,
            dec=dec_simbad,
            radius=CONE_RADIUS_ARCSEC
        )

        if matches is not None and len(matches) > 0:
            ztf_coords  = SkyCoord(
                ra=matches['meanra'].values,
                dec=matches['meandec'].values,
                unit='deg'
            )
            seps        = center_coord.separation(ztf_coords).arcsec
            closest_idx = np.argmin(seps)
            best        = matches.iloc[closest_idx]

            ztf_ids.append(best.get('oid', None))
            ztf_ras.append(float(best['meanra']))
            ztf_decs.append(float(best['meandec']))
            separations.append(float(seps[closest_idx]))

        else:
            ztf_ids.append(None)
            ztf_ras.append(None)
            ztf_decs.append(None)
            separations.append(None)

    except Exception as e:
        print(f"  [ERROR] Failed to cross-match ZTF for {name}: {e}")
        ztf_ids.append(None)
        ztf_ras.append(None)
        ztf_decs.append(None)
        separations.append(None)

    if len(ztf_ids) % 30 == 0:
        print(f"  Progress: {len(ztf_ids)}/{len(valid_df)} processed...")

    time.sleep(0.1)

# 4. Attach columns
valid_df['ZTF_Object_ID']         = ztf_ids
valid_df['ZTF_RA']                = ztf_ras
valid_df['ZTF_DEC']               = ztf_decs
valid_df['SIMBAD_ZTF_Sep_Arcsec'] = separations

# 5. Save
output_path = "BAT_MAXI_SIMBAD_ZTF_perfect_match_copy.csv"
valid_df.to_csv(output_path, index=False)
print(f"\nSuccess! Saved perfectly matched master table → {output_path}")

valid_df[['BAT_Catalogue_Name', 'SIMBAD_ID', 'ZTF_Object_ID', 'SIMBAD_ZTF_Sep_Arcsec']].dropna().head(10)

Loaded 271 sources from: BAT_MAXI_SIMBAD_precise_coords_copy.csv
Found 267 sources with valid coordinates to target on ZTF.

Starting precise ZTF cross-match (Radius: 3.0 arcseconds)...
  Progress: 30/267 processed...
  Progress: 60/267 processed...
  Progress: 90/267 processed...
  Progress: 120/267 processed...
  Progress: 150/267 processed...
  Progress: 180/267 processed...
  Progress: 210/267 processed...
  Progress: 240/267 processed...

Success! Saved perfectly matched master table → BAT_MAXI_SIMBAD_ZTF_perfect_match_copy.csv


,BAT_Catalogue_Name,SIMBAD_ID,ZTF_Object_ID,SIMBAD_ZTF_Sep_Arcsec
2,QSO B0033+595,RX J0035.8+5950,ZTF19abgzjrh,0.133822
4,NGC 262,NGC 262,ZTF18abosnmz,0.663323
5,Mrk 1148,Mrk 1148,ZTF18abtmwft,0.131116
10,4U 0115+634,V* V635 Cas,ZTF18aczqqgn,0.051379
12,1ES 0120+34.0,RX J0123.1+3420,ZTF19acymgje,0.074245
17,QSO J0209+5226,LEDA 138501,ZTF18abjvbhp,0.036735
18,3C 66A,3C 66A,ZTF18aabezmq,0.087575
19,NGC 931,NGC 931,ZTF18acsypzp,0.050086
24,NGC 1275,NGC 1275,ZTF18abvfmot,0.056746
26,GK Per,V* GK Per,ZTF25abxoium,0.074039


In [20]:
# Load the perfect match CSV
df = pd.read_csv("BAT_MAXI_SIMBAD_ZTF_perfect_match_copy.csv")

# Filter: only XRBs (LXB or HXB) with ZTF match
xrb_df = df[
    (df['SIMBAD_Otype'].isin(['LXB', 'HXB'])) &
    (df['ZTF_Object_ID'].notna())
].copy()

print(f"XRBs matched in BAT + MAXI + ZTF: {len(xrb_df)}")

xrb_df.to_csv("XRB_only_BAT_MAXI_ZTF_final.csv", index=False)
print("Saved → XRB_BAT_MAXI_ZTF_final.csv")

xrb_df.head(20)

XRBs matched in BAT + MAXI + ZTF: 20
Saved → XRB_BAT_MAXI_ZTF_final.csv


,BAT_Catalogue_Name,RA_BAT,DEC_BAT,MAXI_Name,SIMBAD_ID,SIMBAD_Otype,SIMBAD_RA,SIMBAD_DEC,ZTF_Object_ID,ZTF_RA,ZTF_DEC,SIMBAD_ZTF_Sep_Arcsec
10,4U 0115+634,19.6300,63.7400,J0118+637,V* V635 Cas,HXB,19.633210,63.742520,ZTF18aczqqgn,19.633232,63.742531,0.051379
53,IGR J06074+2205,91.8500,22.0830,J0607+220,IGR J06074+2205,HXB,91.860884,22.096600,ZTF17aaarcit,91.860871,22.096605,0.046262
54,4U 0614+09,94.2800,9.1370,J0617+091,V* V1055 Ori,LXB,94.280652,9.137088,ZTF18acukluh,94.280684,9.137071,0.128327
55,1A 0620-00,95.6750,-0.3500,J0622-003,1A 0620-00,HXB,95.685593,-0.345636,ZTF18acszzdo,95.685597,-0.345662,0.093497
58,MXB 0656-072,104.6120,-7.2630,J0658-072,2E 1752,HXB,104.572033,-7.209779,ZTF18acpdmgx,104.572026,-7.209791,0.048674
129,Sco X-1,244.9790,-15.6400,J1619-156,V* V818 Sco,LXB,244.979455,-15.640283,ZTF19aaetdak,244.979396,-15.640345,0.303807
140,Her X-1,254.4570,35.3420,J1657+353,V* HZ Her,LXB,254.457546,35.342357,ZTF18aabpzic,254.457540,35.342271,0.312283
192,SAX J1819.3-2525,274.8250,-25.4170,J1819-254,V* V4641 Sgr,HXB,274.840143,-25.407180,ZTF21abvdujh,274.840102,-25.407103,0.308769
196,SAX J1828.5-1037,277.1380,-10.6300,J1828-106,2XMM J182833.7-103700,LXB,277.140731,-10.617697,ZTF18abhjypr,277.140837,-10.617675,0.385012
203,IGR J18410-0535,280.2520,-5.5960,J1841-055,IGR J18410-0535,HXB,280.251814,-5.596243,ZTF19aaonvjw,280.251777,-5.596364,0.455746


In [22]:
df = pd.read_csv("XRB_only_BAT_MAXI_ZTF_final.csv")
print(df.columns.tolist())

['BAT_Catalogue_Name', 'RA_BAT', 'DEC_BAT', 'MAXI_Name', 'SIMBAD_ID', 'SIMBAD_Otype', 'SIMBAD_RA', 'SIMBAD_DEC', 'ZTF_Object_ID', 'ZTF_RA', 'ZTF_DEC', 'SIMBAD_ZTF_Sep_Arcsec']


In [26]:
df = pd.read_csv("XRB_only_BAT_MAXI_ZTF_final.csv")

final = df[[
    'BAT_Catalogue_Name',
    'RA_BAT',
    'DEC_BAT',
    'MAXI_Name',
    'SIMBAD_ID',
    'SIMBAD_Otype',
    'SIMBAD_RA',
    'SIMBAD_DEC',
    'ZTF_Object_ID'
]].copy()
final['SIMBAD_RA']  = final['SIMBAD_RA'].round(4)  # ADD HERE
final['SIMBAD_DEC'] = final['SIMBAD_DEC'].round(4) # ADD HERE

final.to_csv("XRB_BAT_MAXI_ZTF_final_report.csv", index=False)
print(f"Saved {len(final)} XRBs → XRB_BAT_MAXI_ZTF_final.csv")
final.head()

Saved 20 XRBs → XRB_BAT_MAXI_ZTF_final.csv


,BAT_Catalogue_Name,RA_BAT,DEC_BAT,MAXI_Name,SIMBAD_ID,SIMBAD_Otype,SIMBAD_RA,SIMBAD_DEC,ZTF_Object_ID
0,4U 0115+634,19.630,63.740,J0118+637,V* V635 Cas,HXB,19.6332,63.7425,ZTF18aczqqgn
1,IGR J06074+2205,91.850,22.083,J0607+220,IGR J06074+2205,HXB,91.8609,22.0966,ZTF17aaarcit
2,4U 0614+09,94.280,9.137,J0617+091,V* V1055 Ori,LXB,94.2807,9.1371,ZTF18acukluh
3,1A 0620-00,95.675,-0.350,J0622-003,1A 0620-00,HXB,95.6856,-0.3456,ZTF18acszzdo
4,MXB 0656-072,104.612,-7.263,J0658-072,2E 1752,HXB,104.5720,-7.2098,ZTF18acpdmgx
